<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 6 · EODHD API: From Static CSV to Live Data

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook illustrates how the EODHD wrapper is used in the Chapter 6 script and mirrors its diagnostics with locally available data.

- Reuse the `compute_log_returns` helper from `ch06_eodhd_autocorr_demo.py`.
- Compare daily and intraday-style autocorrelations on synthetic data.
- Prepare the mental model needed for live-data experiments with EODHD.


In [ ]:
import sys
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch05_eod_engineering as ch05  # static EOD data helpers
from ch06_eodhd_autocorr_demo import compute_log_returns  # log-return helper


### 1. Daily Log-Return Autocorrelation

We start from the static SPY series and apply the same `compute_log_returns` logic that the EODHD script uses for remote data.

In [ ]:
panel = ch05.load_eod_panel()
spy_close = panel["SPY"].dropna()
daily_log_ret = compute_log_returns(spy_close)

acf1_daily = float(daily_log_ret.autocorr(lag=1))
print(f"Daily lag-1 autocorrelation (SPY static panel): {acf1_daily: .3f}")


### 2. Pseudo Intraday Series via Resampling

To mimic the intraday experiment without contacting the live API, we resample the daily data into a higher-frequency series by linearly interpolating between closes. This keeps the code patterns identical while avoiding network calls.

In [ ]:
daily = spy_close.to_frame("close")
intraday_index = pd.date_range(
    start=daily.index.min(),
    end=daily.index.max(),
    freq="5T",
)
intraday = daily.reindex(intraday_index).interpolate()  # simple smoothing
intraday_log_ret = compute_log_returns(intraday["close"])
acf1_intraday = float(intraday_log_ret.autocorr(lag=1))
print(f"5-minute-style lag-1 autocorrelation (interpolated): {acf1_intraday: .3f}")


### 3. Visual Comparison

The figure below mirrors the Chapter 6 visualisation: a normalised price path and a histogram of daily log-returns.

In [ ]:
fig, (ax_price, ax_hist) = plt.subplots(2, 1, figsize=(7.0, 4.6))

normed_price = spy_close / spy_close.iloc[0]
ax_price.plot(normed_price.index, normed_price.values, color="tab:blue")
ax_price.set_ylabel("price (normalised)")
ax_price.set_title("SPY: static EOD price path")
ax_price.grid(True, alpha=0.3)

ax_hist.hist(daily_log_ret.values, bins=80, color="tab:orange", alpha=0.8)
ax_hist.set_xlabel("daily log-return")
ax_hist.set_ylabel("frequency")
ax_hist.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


### Takeaways

- The EODHD client and the static CSV share the same core diagnostics: log-returns and short-lag autocorrelations.
- You can prototype all logic on static data before wiring in live API calls.
- When you later connect to EODHD, the main change is where prices come from, not how you compute statistics.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">